# Paper 09 · Deep Residual Learning

**Citation:** Kaiming He et al., “Deep Residual Learning for Image Recognition” (2015).

**Paper:** https://arxiv.org/abs/1512.03385

> **Scale gap:** We compare plain and residual deep MLPs on digits instead of ImageNet-scale CNNs.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. What is the degradation problem?
2. Why is a deeper model not guaranteed to train better even if it has more capacity?
3. How can an identity path affect gradient flow?

## Central claim
Residual connections make deeper networks easier to optimize by learning residual functions around identity paths.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
d=load_digits(); X=torch.tensor((d.data/16).astype("float32")); y=torch.tensor(d.target.astype("int64"))
tr,te=train_test_split(np.arange(len(y)),test_size=.3,random_state=0,stratify=y.numpy())
Xtr,Xte,ytr,yte=X[tr],X[te],y[tr],y[te]

## Plain versus residual blocks

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self,d): super().__init__(); self.net=nn.Sequential(nn.Linear(d,d),nn.ReLU())
    def forward(self,x): return self.net(x)

class ResidualBlock(nn.Module):
    def __init__(self,d): super().__init__(); self.net=nn.Sequential(nn.Linear(d,d),nn.ReLU(),nn.Linear(d,d))
    def forward(self,x):
        # TODO: rewrite the residual equation from memory.
        return torch.relu(x+self.net(x))

class DeepNet(nn.Module):
    def __init__(self,depth,residual):
        super().__init__(); D=64
        Block=ResidualBlock if residual else PlainBlock
        self.inp=nn.Sequential(nn.Linear(64,D),nn.ReLU())
        self.blocks=nn.Sequential(*[Block(D) for _ in range(depth)])
        self.out=nn.Linear(D,10)
    def forward(self,x): return self.out(self.blocks(self.inp(x)))

## Train at increasing depth

In [ ]:
def train(depth,residual,epochs=45):
    torch.manual_seed(0); m=DeepNet(depth,residual); opt=torch.optim.Adam(m.parameters(),lr=.003); ce=nn.CrossEntropyLoss(); hist=[]
    for _ in range(epochs):
        opt.zero_grad(); loss=ce(m(Xtr),ytr); loss.backward(); opt.step()
        with torch.no_grad(): acc=(m(Xte).argmax(1)==yte).float().mean().item()
        hist.append([loss.item(),acc])
    return np.array(hist)
rows=[]
for depth in [2,6,12]:
    for residual in [False,True]:
        h=train(depth,residual); rows.append((depth,residual,h[-1,0],h[-1,1]))
        plt.plot(h[:,0],label=f"{'res' if residual else 'plain'} d={depth}")
print(rows)
plt.yscale("log"); plt.legend(); plt.title("Optimization with depth"); plt.show()

### Ablation
Replace the residual addition with concatenation or a learned projection. Does the optimization behavior still look the same?

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this paper?
2. What was actually new?
3. What evidence did your notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which idea survived into modern systems?
6. What would you test next?